# Technical Interoperability 

This notebook demonstrates how one can achieve technical interoperability. It is important to note that in some cases syntatic interoperability is considered separately but in our case we consider as one. 

The example uses a simple scenario where **Company A** shares personal data with **Company B** through a standardized interface.


## Setup

Standard library plus `pandas`, `pyarrow` (Parquet), and `jsonschema`. The cell installs anything missing.

In [1]:
import sys, subprocess

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, name in [("pandas", "pandas"), ("pyarrow", "pyarrow"), ("jsonschema", "jsonschema")]:
    _ensure(pkg, name)

import json, io
from dataclasses import dataclass, asdict
import pandas as pd
import jsonschema

print("Environment ready.")

Environment ready.


## 1. The problem: two systems, two internal models

Consider two organizations:

- **Company A** stores personal data collected from a data subject.
- **Company B** receives the data to provide a service requested by the data subject.

The code below shows the problem that will occur when trying to exchanging the data due to difference in their internal models.


In [2]:
#  Company A internal model  
@dataclass
class PersonalDataRecord:
    subject_id: str
    email: str
    country: str
    data_category: str
    created_date: str
    share_allowed: bool
    retention_until: str

#  Company A internal model  
@dataclass
class AnalyticsRecord:
    record_id: str        # not "data subject_id"
    contact: str          # not "email"
    region: str           # not "country"
    category: str         # not "personal data record_type"
    opened: str           # not "created_date"
    shareable: bool       # not "share_allowed"
    expires: str          # not "retention_until"
    

source_records = [
    {
        "subject_id": "C001",
        "email": "ana@example.com",
        "country": "DE",
        "data_category": "billing",
        "created_date": "2026-01-10",
        "share_allowed": True,
        "retention_until": "2027-01-10",
    },
    {
        "subject_id": "C002",
        "email": "ben@example.com",
        "country": "FR",
        "data_category": "login",
        "created_date": "2026-02-03",
        "share_allowed": False,
        "retention_until": "2027-02-03",
    },
]
producer_records = [PersonalDataRecord(**r) for r in source_records]


# --- Company B internal model ------------------------------------------
@dataclass
class AnalyticsRecord:
    record_id: str
    contact: str
    region: str
    category: str
    opened: str
    shareable: bool
    expires: str

# Direct reuse fails because the models differ
try:
    rec = producer_records[0]
    _ = rec.region
except AttributeError as e:
    print("Direct object reuse fails:", e)

Direct object reuse fails: 'PersonalDataRecord' object has no attribute 'region'


## 2. Technical interoperability

The code below addresses the interoperability challenge discussed above by introducing a shared exchange schema. The schema defines a common structure and validation rules for the personal data exchanged between Company A and Company B. By agreeing on this schema, both organizations can exchange data in a consistent format regardless of their internal data models.

In addition, the schema serves as the foundation for communication through a common protocol. This ensures that exchanged data can be validated, transmitted, and processed reliably by both systems.

In [3]:
import json
from jsonschema import validate

# ------------------------------------------------------------------
# Agreed exchange schema between Company A and Company B
# ------------------------------------------------------------------

exchange_schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "ExchangePersonalDataRecord",
    "type": "object",
    "required": [
        "subject_id",
        "email",
        "country",
        "data_category",
        "created_date",
        "share_allowed",
        "retention_until"
    ],
    "additionalProperties": False,
    "properties": {
        "subject_id": {
            "type": "string",
            "pattern": "^C\\d+$"
        },
        "email": {
            "type": "string",
            "format": "email"
        },
        "country": {
            "type": "string",
            "pattern": "^[A-Z]{2}$"
        },
        "data_category": {
            "type": "string"
        },
        "created_date": {
            "type": "string"
        },
        "share_allowed": {
            "type": "boolean"
        },
        "retention_until": {
            "type": "string"
        }
    }
}

# ------------------------------------------------------------------
# Company A maps its internal model to the agreed exchange format
# ------------------------------------------------------------------

def to_exchange_format(record: PersonalDataRecord) -> dict:
    return {
        "subject_id": record.subject_id,
        "email": record.email,
        "country": record.country,
        "data_category": record.data_category,
        "created_date": record.created_date,
        "share_allowed": record.share_allowed,
        "retention_until": record.retention_until,
    }

# Create exchange payload
exchange_records = [
    to_exchange_format(r)
    for r in producer_records
]

# Validate against agreed schema
for record in exchange_records:
    validate(instance=record, schema=exchange_schema)

print("✓ All records satisfy the agreed schema")

# ------------------------------------------------------------------
# Simulated transfer using JSON over HTTPS/REST
# ------------------------------------------------------------------

payload = json.dumps(exchange_records)

print("\nJSON payload sent by Company A:")
print(payload)

# ------------------------------------------------------------------
# Company B receives and processes the payload
# ------------------------------------------------------------------

received_records = json.loads(payload)

analytics_records = [
    AnalyticsRecord(
        record_id=r["subject_id"],
        contact=r["email"],
        region=r["country"],
        category=r["data_category"],
        opened=r["created_date"],
        shareable=r["share_allowed"],
        expires=r["retention_until"]
    )
    for r in received_records
]

print("\n✓ Successfully imported into Company B")
print("\nFirst AnalyticsRecord:")
print(analytics_records[0])

✓ All records satisfy the agreed schema

JSON payload sent by Company A:
[{"subject_id": "C001", "email": "ana@example.com", "country": "DE", "data_category": "billing", "created_date": "2026-01-10", "share_allowed": true, "retention_until": "2027-01-10"}, {"subject_id": "C002", "email": "ben@example.com", "country": "FR", "data_category": "login", "created_date": "2026-02-03", "share_allowed": false, "retention_until": "2027-02-03"}]

✓ Successfully imported into Company B

First AnalyticsRecord:
AnalyticsRecord(record_id='C001', contact='ana@example.com', region='DE', category='billing', opened='2026-01-10', shareable=True, expires='2027-01-10')
